# Grad-CAM — confronto B2 / Q / QF

Analisi separata dalla tesi e dal training. Nessun file degli esperimenti viene modificato.
Si leggono un archivio di stato e quello del dataset; si estraggono soltanto tre
checkpoint e poche immagini. **Nessun nuovo addestramento e nessun download di galassie.**

La mappa spiega il **logit della classe vera**, mantenendo lo stesso target per tutti
i modelli, anche se le predizioni sono diverse. Il circuito di Q/QF resta nel percorso
del gradiente. Le mappe 7×7 sono interpolate sul crop 224×224: non individuano strutture
con precisione al pixel e non dimostrano un meccanismo causale.

CPU sufficiente per pochi esempi. La prima installazione richiede il download dei
pacchetti e, se assenti, dei pesi ImageNet ResNet18. Tutto l'ambiente scientifico è
isolato in `/content/heat_map/.venv`: non si modifica NumPy nel kernel di Colab,
non serve riavviare il runtime, non si installa JAX.

## 1. Percorsi e selezione

Controlla i due percorsi degli archivi. Per `full` usa `colab_state.tar.gz`;
per 100/300/900 seleziona il tuo `colab_subset_state.tar.gz` e cambia `SUBSET`.
Sono accettate anche directory già estratte, purché contengano le run o il dataset.

Il seed 7 è fissato come esempio, non selezionato come migliore. Si sceglie un'immagine
per classe con una regola deterministica basata sull'ObjID, indipendente dalle predizioni.
L'esempio aggiuntivo di disaccordo è deliberatamente selezionato e viene etichettato.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    WORK_DIR = Path('/content/heat_map')
    BUNDLE_DIR = Path('/content/drive/MyDrive/tesi_qml_colab/colab_bundle')
    STATE_SOURCE = BUNDLE_DIR / 'colab_state.tar.gz'
    DATASET_SOURCE = BUNDLE_DIR / 'colab_dataset.tar.gz'
    SAVE_DIR = Path('/content/drive/MyDrive/tesi_qml_colab/heat_map')
else:
    WORK_DIR = Path.cwd() if Path.cwd().name == 'heat_map' else Path.cwd() / 'heat_map'
    PROJECT = WORK_DIR.parent
    STATE_SOURCE = PROJECT / 'colab_pacchetto_finale/colab_state'
    DATASET_SOURCE = PROJECT / 'colab_pacchetto_finale/stack_and_notebooks/colab_dataset.tar.gz'
    SAVE_DIR = WORK_DIR / 'exports'

TASK = 'three_class'       # 'binary' oppure 'three_class'
SUBSET = 'full'            # '100', '300', '900', 'full'
REPEAT_SEED = 7
QUBITS = 4                # 4 oppure 8
PER_CLASS = 1
EXTRA_DISAGREEMENT = True
SELECTION_SEED = 20260907
MANUAL_OBJIDS = []         # opzionale: stringhe ObjID presenti nel TEST del seed
TRUST_MY_CHECKPOINTS = True  # solo archivi tuoi: torch.load usa pickle
WEIGHTS_PATH = None        # opzionale: resnet18-f37072fd.pth già presente
USE_ISOLATED_ENV = IN_COLAB

WORK_DIR = WORK_DIR.resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
for path in (STATE_SOURCE, DATASET_SOURCE):
    if not path.exists():
        raise FileNotFoundError(f'Correggi il percorso: {path}')
print({'work': str(WORK_DIR), 'state': str(STATE_SOURCE), 'dataset': str(DATASET_SOURCE),
       'task': TASK, 'subset': SUBSET, 'qubits': QUBITS, 'seed': REPEAT_SEED})


## 2. Motore incluso nel notebook

Questa cella salva una copia autonoma del motore **dentro heat_map**, con nome distinto
dal sorgente locale. Non modifica o importa i moduli di `tesi_qml`.

In [ ]:
HELPER_SOURCE = '"""Read-only reconstruction of thesis heads and class-targeted Grad-CAM.\n\nAll caches, extracted inputs and outputs live beside this file in heat_map.\nNo optimizer, training, RNG restoration or modification of the source snapshots.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport sys\nimport tarfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nHERE = Path(__file__).resolve().parent\nsys.dont_write_bytecode = True\nos.environ[\'MPLCONFIGDIR\'] = str(HERE / \'cache/matplotlib\')\nos.environ[\'TORCH_HOME\'] = str(HERE / \'cache/torch\')\nos.environ[\'XDG_CACHE_HOME\'] = str(HERE / \'cache\')\nos.environ[\'MPLBACKEND\'] = \'Agg\'\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn import functional as F\n\nMODELS = (\'B2\', \'Q\', \'QF\')\nMEAN = (.485, .456, .406)\nSTD = (.229, .224, .225)\nLABELS = {\'elliptical\': \'Ellittica\', \'spiral\': \'Spirale\', \'merger\': \'Merger\'}\n\n\ndef safe_name(value):\n    value = str(value).replace(\'\\\\\', \'/\')\n    path = PurePosixPath(value)\n    if path.is_absolute() or \'..\' in path.parts or any(\':\' in p for p in path.parts):\n        raise ValueError(f\'Unsafe archive/path name: {value}\')\n    return str(path)\n\n\nclass Snapshot:\n    """Folder or tar.gz reader. Never invokes extractall or trusts member paths."""\n    def __init__(self, path):\n        self.path = Path(path).expanduser().resolve()\n        self.tar = None\n        if self.path.is_dir():\n            self.members = {safe_name(p.relative_to(self.path).as_posix()): p\n                            for p in self.path.rglob(\'*\') if p.is_file()}\n        elif self.path.is_file():\n            self.tar = tarfile.open(self.path, \'r:*\')\n            self.members = {}\n            try:\n                for member in self.tar.getmembers():\n                    name = safe_name(member.name)\n                    if member.isdir():\n                        continue\n                    if not member.isfile():\n                        raise ValueError(f\'Archive links/special files are not allowed: {name}\')\n                    if name in self.members:\n                        raise ValueError(f\'Duplicate archive member: {name}\')\n                    self.members[name] = member\n            except Exception:\n                self.tar.close()\n                raise\n        else:\n            raise FileNotFoundError(f\'Snapshot not found: {self.path}\')\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *args):\n        if self.tar:\n            self.tar.close()\n\n    def read(self, name):\n        name = safe_name(name)\n        member = self.members[name]\n        if self.tar:\n            with self.tar.extractfile(member) as stream:\n                return stream.read()\n        if not member.resolve().is_relative_to(self.path):\n            raise ValueError(\'Source file points outside snapshot\')\n        return member.read_bytes()\n\n    def suffix(self, suffix):\n        suffix = safe_name(suffix)\n        matches = [n for n in self.members if n == suffix or n.endswith(\'/\' + suffix)]\n        if len(matches) != 1:\n            raise ValueError(f\'Expected one source for {suffix}, found {len(matches)}\')\n        return matches[0]\n\n    def run(self, task, subset, seed, qubits, model):\n        found = []\n        for name in self.members:\n            if not name.endswith(\'run_config.json\'):\n                continue\n            config = json.loads(self.read(name))\n            if (config.get(\'task\') == task and str(config.get(\'subset_size\')) == str(subset)\n                    and config.get(\'repeat_seed\') == seed and config.get(\'n_qubits\') == qubits\n                    and config.get(\'model\') == model and config.get(\'iteration\') == 1):\n                found.append((name, config))\n        if len(found) != 1:\n            raise ValueError(f\'{task}/{subset}/{seed}/{qubits}/{model}: {len(found)} matching runs; select an unambiguous snapshot\')\n        return found[0]\n\n\ndef make_head(config, angle_scale):\n    """Exact final B2/Q/QF ansatz, with checkpoint-compatible state_dict names.\n\n    Mirrors classical.py and PennylaneAngleQNN, but does not import/patch the\n    project. Unsupported architecture signatures are rejected by load_runs.\n    """\n    if angle_scale != \'tanh_pi\':\n        raise ValueError(f\'Unsupported final-matrix angle scale: {angle_scale}\')\n    q = int(config[\'n_qubits\'])\n    layers = int(config[\'n_layers\'])\n    classes = len(config[\'class_names\'])\n    dim = int(config[\'training_config\'][\'feature_bank\'][\'feature_dimension\'])\n    if q not in (4, 8) or layers != 2 or dim != 512:\n        raise ValueError(\'This analysis supports the final 512D, 4/8-qubit, 2-layer matrix only\')\n\n    class B2(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.pre_head = nn.Linear(dim, q)\n            self.classifier = nn.Linear(q, classes)\n\n        def forward(self, features):\n            return self.classifier(torch.tanh(self.pre_head(features)) * np.pi)\n\n    if config[\'model\'] == \'B2\':\n        model = B2()\n    elif config[\'model\'] in (\'Q\', \'QF\'):\n        import pennylane as qml\n\n        @qml.qnode(qml.device(\'default.qubit\', wires=q, shots=None),\n                   interface=\'torch\', diff_method=\'backprop\')\n        def circuit(inputs, weights):\n            for wire in range(q):\n                qml.RY(inputs[..., wire], wires=wire)\n            for layer in range(layers):\n                for wire in range(q):\n                    qml.RZ(weights[layer, wire, 0], wires=wire)\n                    qml.RY(weights[layer, wire, 1], wires=wire)\n                for wire in range(q - 1):\n                    qml.CNOT(wires=[wire, wire + 1])\n            return [qml.expval(qml.PauliZ(wire)) for wire in range(q)]\n\n        class Quantum(nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.pre_quantum = nn.Linear(dim, q)\n                self.quantum_layer = qml.qnn.TorchLayer(circuit, {\'weights\': (layers, q, 2)})\n                self.post_quantum = nn.Linear(q, classes)\n\n            def forward(self, features):\n                angles = torch.tanh(self.pre_quantum(features)) * np.pi\n                values = self.quantum_layer(angles).to(dtype=features.dtype)\n                return self.post_quantum(values)\n\n        model = Quantum()\n    else:\n        raise ValueError(\'Only B2, Q and QF are supported\')\n    # Attribution differentiates inputs, not trained weights; no optimization.\n    model.eval()\n    model.requires_grad_(False)\n    return model\n\n\ndef gradcam_from_spatial(spatial, head, target):\n    """Gradient of a fixed class logit w.r.t. post-layer4 activations.\n\n    Detaching here is exact for d(logit)/d(layer4): GAP and the whole head\n    remain in the graph, including the quantum circuit for Q and QF.\n    """\n    if spatial.ndim != 4 or spatial.shape[0] != 1:\n        raise ValueError(\'One [1,C,H,W] activation tensor is required\')\n    with torch.enable_grad():\n        activation = spatial.detach().clone().requires_grad_(True)\n        features = F.adaptive_avg_pool2d(activation, (1, 1)).flatten(1)\n        logits = head(features)\n        if target < 0 or target >= logits.shape[1]:\n            raise ValueError(\'Target class is outside the classifier\')\n        gradient, = torch.autograd.grad(logits[0, target], activation)\n        weights = gradient.mean(dim=(2, 3), keepdim=True)\n        signed = (weights * activation).sum(dim=1)[0]\n        cam = signed.relu()\n    if not torch.isfinite(cam).all() or not torch.isfinite(gradient).all():\n        raise ValueError(\'Non-finite attribution; no figure will be accepted\')\n    return {\'cam\': cam.detach().cpu().numpy(),\n            \'signed_cam\': signed.detach().cpu().numpy(),\n            \'gradient\': gradient.detach().cpu().numpy(),\n            \'logits\': logits.detach().cpu().numpy()[0],\n            \'probabilities\': logits.detach().double().softmax(dim=1).cpu().numpy()[0]}\n\n\ndef normalize_maps(maps):\n    if any(not np.isfinite(m).all() or (m < 0).any() for m in maps):\n        raise ValueError(\'Grad-CAM display expects finite positive contributions\')\n    maximum = max(float(np.max(m)) for m in maps)\n    return ([m / maximum if maximum > 0 else np.zeros_like(m) for m in maps], maximum)\n\n\ndef check_prediction(actual, saved, tolerance=1e-4):\n    actual, saved = np.asarray(actual, float), np.asarray(saved, float)\n    if actual.shape != saved.shape or not np.isfinite(actual).all() or not np.isfinite(saved).all():\n        raise ValueError(\'Invalid reconstructed/saved probabilities\')\n    difference = float(np.max(np.abs(actual - saved)))\n    if difference > tolerance or np.argmax(actual) != np.argmax(saved):\n        raise ValueError(f\'Prediction mismatch: max probability error={difference:.6g}, tolerance={tolerance}; check weights, preprocessing and checkpoint\')\n    return difference\n\n\ndef select_examples(frame, classes, per_class, selection_seed, extra_disagreement):\n    frame = frame.copy()\n    frame[\'objid\'] = frame.objid.astype(str)\n    if frame.objid.duplicated().any():\n        raise ValueError(\'Duplicate test objid\')\n    frame[\'_selection_hash\'] = [hashlib.sha256(f\'{selection_seed}|{oid}\'.encode()).hexdigest()\n                                for oid in frame.objid]\n    selected = []\n    for name in classes:\n        group = frame[frame.true_class.eq(name)].sort_values([\'_selection_hash\', \'objid\'])\n        if len(group) < per_class:\n            raise ValueError(f\'Not enough test examples for {name}\')\n        selected.append(group.head(per_class).assign(selection_reason=\'hash_sample_by_true_class\'))\n    result = pd.concat(selected, ignore_index=True)\n    if extra_disagreement:\n        disagreement = frame[[f\'pred_{m}\' for m in MODELS]].nunique(axis=1).gt(1)\n        candidates = frame[disagreement & ~frame.objid.isin(result.objid)]\n        # Explicitly selected difficult case, never described as representative.\n        if not candidates.empty:\n            extra = candidates.sort_values([\'_selection_hash\', \'objid\']).head(1)\n            result = pd.concat([result, extra.assign(selection_reason=\'selected_model_disagreement\')], ignore_index=True)\n    return result.drop(columns=\'_selection_hash\')\n\n\ndef load_runs(snapshot, args, output):\n    configs, heads, predictions, audit = {}, {}, {}, {}\n    for model in MODELS:\n        name, config = snapshot.run(args.task, args.subset, args.seed, args.qubits, model)\n        prefix = str(PurePosixPath(name).parent)\n        metrics = json.loads(snapshot.read(prefix + \'/metrics.json\'))\n        if model != \'B2\':\n            signature = metrics.get(\'model_signature\', {})\n            expected = {\'encoding\': \'RY\', \'entangling_gate\': \'CNOT\',\n                        \'entangling_pairs\': [[i, i+1] for i in range(args.qubits-1)],\n                        \'trainable_gates_per_layer\': [\'RZ\', \'RY\'],\n                        \'observables\': [f\'Z{i}\' for i in range(args.qubits)],\n                        \'weight_shape\': [2, args.qubits, 2], \'angle_scale\': \'tanh_pi\'}\n            if any(signature.get(k) != v for k, v in expected.items()):\n                raise ValueError(f\'Unsupported circuit signature for {model}: {signature}\')\n        provenance = config[\'feature_bank_provenance\'][\'test\']\n        if (provenance[\'backbone\'] != \'torchvision.ResNet18_Weights.IMAGENET1K_V1\'\n                or provenance[\'transform\'] != \'eval_center_crop_imagenet_v1\'\n                or provenance[\'views\'] != 1):\n            raise ValueError(f\'Unsupported feature pipeline for {model}\')\n        checkpoint_bytes = snapshot.read(prefix + \'/checkpoints/best.pt\')\n        # Only use archives created by the user: legacy checkpoints include RNG\n        # objects and require pickle deserialization, never restore those RNGs.\n        checkpoint = torch.load(io.BytesIO(checkpoint_bytes), map_location=\'cpu\', weights_only=False)\n        fingerprint = checkpoint[\'extra\'][\'run_metadata\'][\'config_fingerprint\']\n        if fingerprint != config[\'config_fingerprint\']:\n            raise ValueError(f\'Checkpoint/config fingerprint mismatch for {model}\')\n        head = make_head(config, \'tanh_pi\').to(args.device)\n        head.load_state_dict(checkpoint[\'model_state\'], strict=True)\n        head.eval()\n        frame = pd.read_parquet(io.BytesIO(snapshot.read(prefix + \'/test_predictions.parquet\')))\n        frame[\'objid\'] = frame.objid.astype(str)\n        if not frame.objid.str.fullmatch(r\'\\d+\').all():\n            raise ValueError(\'Invalid SDSS object identifier\')\n        if frame.objid.duplicated().any():\n            raise ValueError(f\'Multiple test predictions for one object in {model}\')\n        expected_meta = {\'model\': model, \'task\': args.task, \'repeat_seed\': args.seed,\n                         \'n_qubits\': args.qubits, \'config_fingerprint\': config[\'config_fingerprint\']}\n        if any(not frame[k].eq(v).all() for k, v in expected_meta.items()):\n            raise ValueError(f\'Prediction metadata does not match run {model}\')\n        if not frame.subset_size.astype(str).eq(args.subset).all():\n            raise ValueError(\'Prediction subset mismatch\')\n        configs[model], heads[model], predictions[model] = config, head, frame.set_index(\'objid\', drop=False)\n        input_dir = output / \'inputs\' / model\n        input_dir.mkdir(parents=True)\n        (input_dir / \'run_config.json\').write_bytes(snapshot.read(name))\n        (input_dir / \'best.pt\').write_bytes(checkpoint_bytes)\n        audit[model] = {\'source\': str(snapshot.path), \'run\': prefix, \'checkpoint_epoch\': checkpoint[\'epoch\'],\n                        \'checkpoint_sha256\': hashlib.sha256(checkpoint_bytes).hexdigest(),\n                        \'config_fingerprint\': fingerprint}\n        print(f\'Loaded {model}: best epoch {checkpoint["epoch"]}, {len(frame)} test objects\', flush=True)\n    base = configs[\'B2\']\n    for model in MODELS[1:]:\n        for field in [\'class_names\', \'class_to_index\', \'initialization_checksums\', \'feature_bank_provenance\']:\n            if configs[model][field] != base[field]:\n                raise ValueError(f\'Unpaired {model}/B2 field: {field}\')\n        if set(predictions[model].index) != set(predictions[\'B2\'].index):\n            raise ValueError(\'Test object sets differ between heads\')\n        for field in [\'true_class\', \'class_index\', \'sha256\']:\n            other = predictions[model].reindex(predictions[\'B2\'].index)[field]\n            if not other.eq(predictions[\'B2\'][field]).all():\n                raise ValueError(f\'Test identity/label mismatch: {field}\')\n    common = predictions[\'B2\'][[\'objid\', \'true_class\', \'class_index\', \'path\', \'sha256\']].copy()\n    for model in MODELS:\n        common[f\'pred_{model}\'] = predictions[model].predicted_class\n    return configs, heads, predictions, common.reset_index(drop=True), audit\n\n\ndef load_backbone(args):\n    from torchvision.models import resnet18, ResNet18_Weights\n    if args.weights:\n        weights_path = Path(args.weights).expanduser().resolve()\n    else:\n        weights_path = Path(torch.hub.get_dir()) / \'checkpoints/resnet18-f37072fd.pth\'\n        if not weights_path.is_file():\n            print(\'Downloading the official ImageNet ResNet18 weights into heat_map/cache (no galaxy downloads).\', flush=True)\n            torch.hub.load_state_dict_from_url(ResNet18_Weights.IMAGENET1K_V1.url,\n                                               progress=True, check_hash=True, map_location=\'cpu\')\n    digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()\n    if not digest.startswith(\'f37072fd\'):\n        raise ValueError(\'ResNet18 file is not the expected IMAGENET1K_V1 weight artifact\')\n    model = resnet18(weights=None)\n    model.load_state_dict(torch.load(weights_path, map_location=\'cpu\', weights_only=True), strict=True)\n    model.fc = nn.Identity()\n    model.requires_grad_(False)\n    return model.eval().to(args.device), digest\n\n\ndef spatial_features(backbone, tensor):\n    with torch.no_grad():\n        x = backbone.maxpool(backbone.relu(backbone.bn1(backbone.conv1(tensor))))\n        x = backbone.layer4(backbone.layer3(backbone.layer2(backbone.layer1(x))))\n    return x\n\n\ndef load_image(snapshot, row, output):\n    from torchvision import transforms\n    original_path = str(row[\'path\']).replace(\'\\\\\', \'/\')\n    if \'/data/images/\' in original_path:\n        suffix = \'data/images/\' + original_path.split(\'/data/images/\', 1)[1]\n    elif original_path.startswith(\'data/images/\'):\n        suffix = original_path\n    else:\n        raise ValueError(f\'Unsupported recorded image path: {original_path}\')\n    name = snapshot.suffix(suffix)\n    content = snapshot.read(name)\n    digest = hashlib.sha256(content).hexdigest()\n    if digest != row[\'sha256\']:\n        raise ValueError(f\'Image checksum mismatch for {row["objid"]}\')\n    with Image.open(io.BytesIO(content)) as image:\n        if min(image.size) < 224:\n            raise ValueError(\'Image is smaller than the expected crop\')\n        crop = transforms.CenterCrop(224)(image.convert(\'RGB\'))\n    tensor = transforms.Normalize(MEAN, STD)(transforms.ToTensor()(crop)).unsqueeze(0)\n    image_dir = output / \'inputs/images\'\n    image_dir.mkdir(parents=True, exist_ok=True)\n    (image_dir / f\'{row["objid"]}.jpg\').write_bytes(content)\n    crop.save(image_dir / f\'{row["objid"]}_crop.png\')\n    return crop, tensor, digest\n\n\ndef plot_example(crop, row, results, classes, destination, context=\'\'):\n    import matplotlib.pyplot as plt\n    from matplotlib.cm import ScalarMappable\n    from matplotlib.colors import Normalize\n    resized = [F.interpolate(torch.tensor(results[m][\'cam\'])[None, None], (224, 224),\n                              mode=\'bilinear\', align_corners=False)[0, 0].numpy() for m in MODELS]\n    normalized, scale = normalize_maps(resized)\n    with plt.rc_context({\'font.family\': \'DejaVu Sans\', \'font.size\': 11,\n                         \'pdf.fonttype\': 42, \'savefig.dpi\': 250}):\n        fig, axes = plt.subplots(1, 4, figsize=(13.8, 5.0))\n        fig.subplots_adjust(left=.02, right=.97, bottom=.18, top=.70, wspace=.08)\n        for ax in axes:\n            ax.imshow(crop)\n            ax.axis(\'off\')\n        axes[0].set_title(\'Immagine di test\\nCrop effettivo 224 × 224\', fontsize=11)\n        target = classes.index(row[\'true_class\'])\n        for ax, model, heat in zip(axes[1:], MODELS, normalized):\n            result = results[model]\n            ax.imshow(heat, cmap=\'inferno\', vmin=0, vmax=1, alpha=.70*heat)\n            predicted = int(np.argmax(result[\'probabilities\']))\n            p_target = result[\'probabilities\'][target]\n            color = \'#166534\' if classes[predicted] == row[\'true_class\'] else \'#9F1239\'\n            ax.set_title(f\'{model} · pred.: {LABELS[classes[predicted]]}\\n\'\n                         f\'p(classe spiegata) = {p_target:.3f}\', color=color, fontsize=11)\n            if not np.any(result[\'cam\'] > 0):\n                ax.text(.5, .04, \'Nessun contributo positivo\', transform=ax.transAxes,\n                        ha=\'center\', fontsize=8, color=\'white\', bbox={\'facecolor\':\'black\',\'alpha\':.7})\n        reason = (\'campione deterministico per classe\' if row[\'selection_reason\']==\'hash_sample_by_true_class\'\n                  else \'caso selezionato di disaccordo\' if row[\'selection_reason\']==\'selected_model_disagreement\'\n                  else \'esempio scelto manualmente\')\n        fig.suptitle(f\'{context}\\n\'\n                     f\'Classe vera e spiegata: {LABELS[row["true_class"]]} · ObjID {row["objid"]}\\n\'\n                     f\'{reason}\', y=.98, fontsize=12)\n        bar_ax = fig.add_axes([.36, .09, .38, .025])\n        bar = fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap=\'inferno\'), cax=bar_ax, orientation=\'horizontal\')\n        bar.set_label(\'Contributo positivo relativo · scala comune ai tre modelli per questa immagine\', fontsize=9)\n        paths = {}\n        try:\n            for suffix in (\'png\', \'pdf\'):\n                path = destination.with_suffix(\'.\' + suffix)\n                fig.savefig(path, bbox_inches=\'tight\', facecolor=\'white\')\n                paths[suffix] = str(path)\n        finally:\n            plt.close(fig)\n    return paths, scale\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--state\', required=True, type=Path, help=\'Snapshot folder or trusted state tar.gz\')\n    parser.add_argument(\'--dataset\', required=True, type=Path, help=\'Dataset folder or colab_dataset.tar.gz\')\n    parser.add_argument(\'--task\', choices=[\'binary\', \'three_class\'], default=\'three_class\')\n    parser.add_argument(\'--subset\', choices=[\'100\', \'300\', \'900\', \'full\'], default=\'full\')\n    parser.add_argument(\'--seed\', type=int, default=7)\n    parser.add_argument(\'--qubits\', type=int, choices=[4, 8], default=4)\n    parser.add_argument(\'--per-class\', type=int, default=1)\n    parser.add_argument(\'--selection-seed\', type=int, default=20260907)\n    parser.add_argument(\'--objids\', default=\'\', help=\'Optional comma-separated test IDs; overrides sampling\')\n    parser.add_argument(\'--extra-disagreement\', action=\'store_true\')\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\'], default=\'cpu\')\n    parser.add_argument(\'--weights\', type=Path, help=\'Existing official ResNet18 IMAGENET1K_V1 checkpoint (read-only)\')\n    parser.add_argument(\'--probability-tolerance\', type=float, default=1e-4)\n    parser.add_argument(\'--threads\', type=int, default=2)\n    parser.add_argument(\'--trust-checkpoints\', action=\'store_true\', help=\'Allow pickle loading of your own training checkpoints\')\n    args = parser.parse_args()\n    if not args.trust_checkpoints:\n        parser.error(\'Use --trust-checkpoints only for your own trusted training archives; torch.load uses pickle\')\n    if args.per_class < 1 or args.threads < 1 or args.probability_tolerance <= 0:\n        parser.error(\'per-class, threads and tolerance must be positive\')\n    if args.device == \'cuda\' and not torch.cuda.is_available():\n        parser.error(\'CUDA is not available: select cpu\')\n    torch.set_num_threads(args.threads)\n    torch.manual_seed(0)\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')\n    output = HERE / \'outputs\' / f\'{args.task}_{args.subset}_q{args.qubits}_seed{args.seed}_{stamp}\'\n    output.mkdir(parents=True, exist_ok=False)\n    report = {\'status\': \'started\', \'output\': str(output), \'settings\': {k:str(v) if isinstance(v,Path) else v for k,v in vars(args).items()},\n              \'method\': \'positive Grad-CAM of the TRUE class LOGIT at ResNet18 layer4 output\',\n              \'normalization\': \'shared maximum of positive upsampled CAMs within each image; not across images\',\n              \'spatial_resolution\': \'7x7 interpolated to 224x224; interpolation does not add spatial detail\',\n              \'examples\': []}\n    report_path = output / \'audit.json\'\n    try:\n        import torchvision, pennylane\n        report[\'environment\'] = {\'python\':sys.version, \'torch\':torch.__version__, \'torchvision\':torchvision.__version__,\n                                  \'pennylane\':pennylane.__version__, \'numpy\':np.__version__, \'pandas\':pd.__version__}\n        with Snapshot(args.state) as snapshot:\n            configs, heads, predictions, common, run_audit = load_runs(snapshot, args, output)\n        report[\'runs\'] = run_audit\n        classes = configs[\'B2\'][\'class_names\']\n        if args.objids:\n            ids = [s.strip() for s in args.objids.split(\',\')]\n            if len(ids) != len(set(ids)) or not set(ids).issubset(set(common.objid)):\n                raise ValueError(\'Manual IDs must be unique and present in the shared test set\')\n            selected = common.set_index(\'objid\', drop=False).loc[ids].reset_index(drop=True)\n            selected[\'selection_reason\'] = \'manual\'\n        else:\n            selected = select_examples(common, classes, args.per_class, args.selection_seed, args.extra_disagreement)\n        selected.to_csv(output / \'selected_examples.csv\', index=False)\n        backbone, weight_hash = load_backbone(args)\n        report[\'backbone_sha256\'] = weight_hash\n        (output / \'figures\').mkdir()\n        (output / \'maps\').mkdir()\n        with Snapshot(args.dataset) as dataset:\n            for index, row in enumerate(selected.to_dict(\'records\'), 1):\n                print(f\'[{index}/{len(selected)}] {row["true_class"]} · {row["objid"]}\', flush=True)\n                crop, tensor, image_hash = load_image(dataset, row, output)\n                spatial = spatial_features(backbone, tensor.to(args.device))\n                if tuple(spatial.shape) != (1, 512, 7, 7):\n                    raise ValueError(f\'Unexpected spatial activation shape {spatial.shape}\')\n                target = classes.index(row[\'true_class\'])\n                results, per_model = {}, {}\n                for model in MODELS:\n                    result = gradcam_from_spatial(spatial, heads[model], target)\n                    saved = predictions[model].loc[row[\'objid\']]\n                    probabilities = [saved[f\'prob_{name}_model\'] for name in classes]\n                    error = check_prediction(result[\'probabilities\'], probabilities, args.probability_tolerance)\n                    stored_logits = np.array([saved[f\'logit_{name}\'] for name in classes])\n                    logit_error = float(np.max(np.abs(result[\'logits\']-stored_logits)))\n                    # Softmax is shift invariant: also check logits to protect the\n                    # exact class-score interpretation used by this attribution.\n                    if logit_error > .001:\n                        raise ValueError(f\'Logit mismatch for {model}: {logit_error}\')\n                    results[model] = result\n                    np.savez_compressed(output / \'maps\' / f\'{row["objid"]}_{model}.npz\',\n                                        **result, objid=row[\'objid\'], target_class=row[\'true_class\'])\n                    per_model[model] = {\'probabilities\':result[\'probabilities\'].tolist(),\n                        \'predicted_class\':classes[int(np.argmax(result[\'probabilities\']))],\n                        \'max_probability_error\':error, \'max_logit_error\':logit_error,\n                        \'positive_cam_max\':float(result[\'cam\'].max()),\n                        \'zero_positive_cam\':bool(not np.any(result[\'cam\']>0)),\n                        \'gradient_abs_max\':float(np.abs(result[\'gradient\']).max())}\n                    print(f\'  {model}: prediction verified (max probability error {error:.2g})\', flush=True)\n                task_label = \'Binario\' if args.task == \'binary\' else \'Tre classi\'\n                context = f\'{task_label} · {args.subset.capitalize()} · {args.qubits} dimensioni/qubit · seed {args.seed}\'\n                paths, maximum = plot_example(crop, row, results, classes, output/\'figures\'/f\'{index:02d}_{row["true_class"]}_{row["objid"]}\', context)\n                report[\'examples\'].append({**row, \'image_sha256\':image_hash, \'models\':per_model,\n                                            \'display_maximum\':maximum, \'figures\':paths})\n                report_path.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n        report[\'status\'] = \'complete\'\n        print(f\'COMPLETE: {output}\', flush=True)\n    except Exception as error:\n        report[\'status\'] = \'failed\'\n        report[\'error\'] = f\'{type(error).__name__}: {error}\'\n        raise\n    finally:\n        report_path.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n'

REQUIREMENTS = '# Installed ONLY in heat_map/.venv; torch/torchvision from the CPU wheel index.\nnumpy==2.2.6\npandas==2.3.3\nscipy==1.15.3\npennylane==0.42.3\nautoray==0.7.2\nmatplotlib==3.10.6\nPillow==11.3.0\npyarrow==21.0.0\n# JAX is not needed for the Torch/backprop QNode. Do not install CUDA JAX plugins.\n'
ENGINE = WORK_DIR / 'gradcam_colab_engine.py'
ENGINE.write_text(HELPER_SOURCE, encoding='utf-8')
REQ_FILE = WORK_DIR / 'requirements_colab_generated.txt'
REQ_FILE.write_text(REQUIREMENTS, encoding='utf-8')
print('Motore salvato in', ENGINE)


## 3. Ambiente isolato

Non eseguire installazioni nel runtime principale. L'installazione seguente usa una
virtualenv **senza accesso ai pacchetti globali**, così JAX e i plugin preinstallati di
Colab non interferiscono. Alla prima esecuzione serve connettività per i pacchetti.
Il runtime Colab non è stato eseguito da questo notebook durante la preparazione locale.

In [ ]:
import venv
ENV = dict(os.environ)
for name, folder in [('MPLCONFIGDIR','cache/matplotlib'), ('TORCH_HOME','cache/torch'),
                     ('XDG_CACHE_HOME','cache'), ('TMPDIR','tmp'), ('TEMP','tmp'), ('TMP','tmp')]:
    destination = WORK_DIR / folder
    destination.mkdir(parents=True, exist_ok=True)
    ENV[name] = str(destination)
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
ENV['MPLBACKEND'] = 'Agg'

def live(command):
    print('$', ' '.join(map(str, command)), flush=True)
    with subprocess.Popen(list(map(str, command)), cwd=WORK_DIR, env=ENV,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

if USE_ISOLATED_ENV:
    env_dir = WORK_DIR / '.venv'
    PYTHON = env_dir / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    if not PYTHON.exists():
        venv.EnvBuilder(with_pip=True, system_site_packages=False).create(env_dir)
    pip = [PYTHON, '-m', 'pip', 'install', '--cache-dir', WORK_DIR / 'cache/pip']
    live(pip + ['torch==2.8.0', 'torchvision==0.23.0',
                '--index-url', 'https://download.pytorch.org/whl/cpu'])
    live(pip + ['-r', REQ_FILE])
else:
    PYTHON = Path(sys.executable)
    print('Uso ambiente locale esistente, senza installare o modificare pacchetti.')

live([PYTHON, '-B', '-c',
      "import torch,torchvision,numpy,pandas,pennylane; "
      "print({'torch':torch.__version__,'torchvision':torchvision.__version__,"
      "'numpy':numpy.__version__,'pandas':pandas.__version__,'pennylane':pennylane.__version__})"])


## 4. Generazione e salvataggio

Prima di accettare una figura vengono verificati: identità delle run, checkpoint,
test condiviso, SHA-256 dell'immagine, probabilità e classe predetta rispetto alle
predizioni archiviate. Tolleranza sulle probabilità: 0,0001; sui logit: 0,001.
**Un disallineamento interrompe l'analisi: non allargare la tolleranza per aggirarlo.**

Ogni esecuzione crea una cartella con timestamp. Dopo il processo viene salvato uno
ZIP in `SAVE_DIR`, anche in caso di errore (con stato `failed`, non pronto per la tesi).
I file `.pt` sono checkpoint propri e fidati: non usare archivi di provenienza sconosciuta.

In [ ]:
if not TRUST_MY_CHECKPOINTS:
    raise RuntimeError('Conferma di usare soltanto i tuoi checkpoint fidati.')

command = [PYTHON, '-B', '-u', ENGINE,
           '--state', STATE_SOURCE, '--dataset', DATASET_SOURCE,
           '--task', TASK, '--subset', SUBSET, '--seed', REPEAT_SEED,
           '--qubits', QUBITS, '--per-class', PER_CLASS,
           '--selection-seed', SELECTION_SEED, '--device', 'cpu', '--trust-checkpoints']
if EXTRA_DISAGREEMENT:
    command += ['--extra-disagreement']
if MANUAL_OBJIDS:
    command += ['--objids', ','.join(map(str, MANUAL_OBJIDS))]
if WEIGHTS_PATH:
    command += ['--weights', WEIGHTS_PATH]

before = set((WORK_DIR / 'outputs').glob('*')) if (WORK_DIR / 'outputs').exists() else set()
LAST_OUTPUT = None
try:
    live(command)
finally:
    generated = set((WORK_DIR / 'outputs').glob('*')) - before
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    for folder in sorted(generated):
        LAST_OUTPUT = folder
        archive = shutil.make_archive(str(SAVE_DIR / folder.name), 'zip', folder)
        print('Salvato:', archive)

if LAST_OUTPUT is None:
    raise RuntimeError('Il processo non ha prodotto una cartella di analisi.')
AUDIT = json.loads((LAST_OUTPUT / 'audit.json').read_text(encoding='utf-8'))
if AUDIT['status'] != 'complete':
    raise RuntimeError('Analisi non completa: leggere audit.json, non usare le figure nella tesi.')
print({'status': AUDIT['status'], 'examples': len(AUDIT['examples']), 'output': str(LAST_OUTPUT)})


## 5. Anteprima e lettura

Ogni figura confronta B2, Q e QF sullo **stesso target**. Per un errore non stiamo
automaticamente spiegando la classe sbagliata predetta: spieghiamo sempre quella vera.
La barra usa il massimo positivo comune ai tre pannelli della stessa immagine;
non confrontare l'intensità fra immagini né interpretarla come probabilità o importanza
causale. Una mappa vuota significa assenza di contributi positivi secondo questa regola,
non assenza generale di informazione. I valori grezzi, anche firmati, sono salvati nei `.npz`.

Per cambiare 4→8 qubit basta modificare `QUBITS` e ripetere la cella di generazione.
Per i subset cambiare anche archivio e `SUBSET`. Nessun training viene riavviato.

In [ ]:
from IPython.display import display, Image as DisplayImage
if AUDIT['status'] != 'complete':
    raise RuntimeError('Mostrare soltanto analisi complete.')
for path in sorted((LAST_OUTPUT / 'figures').glob('*.png')):
    display(DisplayImage(filename=str(path), width=1200))


## Uso nella tesi

- Capitolo 4: metodo Grad-CAM e un esempio esplicitamente illustrativo.
- Capitolo 5: confronto qualitativo, senza inferire superiorità globale da un seed.
- Non aggiornare automaticamente i capitoli o la bibliografia principale.
- PDF per Overleaf in `figures/`; `audit.json` e `selected_examples.csv` conservano
  la provenienza; `maps/` conserva le mappe grezze e i gradienti.

Riferimento: Selvaraju et al., *Grad-CAM: Visual Explanations From Deep Networks
via Gradient-Based Localization*, ICCV 2017, pp. 618–626.
[Articolo originale](https://openaccess.thecvf.com/content_iccv_2017/html/Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.html).